In [7]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

JOBLIB_PATH = Path("output/modelo_priorizacao_alertas.joblib")

artifact = joblib.load(JOBLIB_PATH)
pipe = artifact["pipeline"]

print("classes:", artifact.get("classes_") or list(pipe.classes_))
print("features:", artifact["feature_names"])
print("métricas (treino exploratório):", artifact.get("metrics"))

classes: ['alta', 'baixa', 'media']
features: ['area_alerta_ha', 'area_imovel_ha', 'modulos_fiscais', 'fracao_imovel', 'recencia_dias', 'n_alertas_imovel', 'status_alerta', 'fonte_alerta', 'status_imovel', 'condicao', 'tipo_imovel', 'uf']
métricas (treino exploratório): {'f1_macro': 0.807833636180093, 'n_rows': 639, 'n_train': 447, 'n_test': 192, 'exploratory': True}


In [8]:
def prever(df: pd.DataFrame) -> pd.DataFrame:
    X = df[artifact["feature_names"]].copy()
    classes = list(pipe.classes_)
    proba = pipe.predict_proba(X)
    out = df.copy()
    out["prioridade"] = pipe.predict(X)
    for i, cls in enumerate(classes):
        out[f"proba_{cls}"] = proba[:, i]
    if "alta" in classes:
        out = out.sort_values("proba_alta", ascending=False)
    return out.reset_index(drop=True)

In [9]:
test = pd.DataFrame(
    [
        {
            "area_alerta_ha": 24.05,
            "area_imovel_ha": 500.0,
            "modulos_fiscais": 5.0,
            "fracao_imovel": 24.05 / 500.0,
            "recencia_dias": 30.0,
            "n_alertas_imovel": 6,
            "status_alerta": "published",
            "fonte_alerta": "DETER",
            "status_imovel": "AT",
            "condicao": "Analisado, aguardando atendimento a notificação",
            "tipo_imovel": "IRU",
            "uf": "AC",
        }
    ]
)


In [10]:
df = prever(test)

In [11]:
df

,area_alerta_ha,area_imovel_ha,modulos_fiscais,fracao_imovel,recencia_dias,n_alertas_imovel,status_alerta,fonte_alerta,status_imovel,condicao,tipo_imovel,uf,prioridade,proba_alta,proba_baixa,proba_media
0,24.05,500.0,5.0,0.0481,30.0,6,published,DETER,AT,"Analisado, aguardando atendimento a notificação",IRU,AC,alta,0.907932,0.028652,0.063416
